In [2]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [3]:
####### load common directories and data
time_interval = 10 #sec/frame
whichpcs = [1,2]
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_shape/')
datadir = basedir.joinpath('Data_and_Figs')
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
ttot = time_interval * 180
ntrans = 1
bsiter = 3000

In [4]:
### restrict data to RANDOM
treatments = ['Random']

savedir = basedir.joinpath('random')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment.isin(treatments)].copy()

In [7]:
########### all drugs together
treatments = ['DMSO','CK666','Para-Nitro-Blebbistatin']

savedir = basedir.joinpath('drug')
if not savedir.exists():
    savedir.mkdir()

#limit data to the CK666 experiments
TotalFrame = FullFrame[FullFrame.Treatment.isin(treatments)].copy()
TotalFrame['Treatment'] = pd.Categorical(TotalFrame.Treatment.to_list(), categories=treatments, ordered=True)


In [22]:
### restrict data to galvanotaxis experiments
treatments = ['Galvanotaxis']

savedir = basedir.joinpath('galv/')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment.isin(treatments)].copy()

In [3]:
### Don't restrict dataframe at all, calculate detailed balance for all confocal data
savedir = basedir.joinpath('all_experiments/')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame.copy()

In [6]:
############# create all CGPSs #############

binlist = [i for i in TotalFrame.columns.to_list() if 'bin' in i]
allsavedir = savedir.joinpath('allCGPS')
if not allsavedir.exists():
    allsavedir.mkdir(parents = True)
for a in range(1,len(binlist)+1):
    for b in range(1,len(binlist)+1):
        if a == b:
            continue
        elif allsavedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this CGPS')
            continue
        else:
            #set the PCs
            abwhichpcs = [a,b]
            if __name__ ==  '__main__':
                
                ########### get raw transitions and pairs ###########
                rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                        TotalFrame, #pandas dataframe with all of the cgps binned data
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        allsavedir, #where to save the aggregated trajectories
                        )
                
                ########### interpolate all transitions so that only individual transitions are made ###########
                transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                        rawtrans, #pandas dataframe with all of the cgps binned data
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        allsavedir, #where to save the aggregated trajectories
                        )

                ############## get the counts of cells leaving 
                trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                        transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        )
                
                
            

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1446.607706349432 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1436.3095298891096 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1440.2516160128548 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1431.2833464220341 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1448.7206903813883 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1459.773782038232 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
T

In [5]:
########### calculate all the aers and cfs around all the pairwise cgps ###############

binlist = [i for i in TotalFrame.columns.to_list() if 'bin' in i]
allsavedir = savedir.joinpath('allCGPS')
if not allsavedir.exists():
    allsavedir.mkdir(parents = True)
    
#### all the cgps origins determinned by visual inspection (specifically for random treatment)
allorigins = [[[8,8],[7,9],[8,8],[8,8],[8,8],[8,8],[9,7]],
                [[9,6],[9,9],[8,8],[8,8],[8,7],[8,8]],
                    [[8,8],[7,8],[8,8],[8,8],[9,8]],
                        [[9,8],[8,8],[8,8],[8,8]],
                            [[8,9],[8,8],[8,8]],
                                [[8,7],[8,7]],
                                    [[8,8]]]
    
for a in range(1,len(binlist)+1):
    for b in range(1,len(binlist)+1):
        if a == b:
            continue
        elif allsavedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this plot')
            continue
        else:
            #set the PCs
            abwhichpcs = [a,b]
            if __name__ ==  '__main__':
                
                #### open the transitions
                rawtrans = pd.read_csv(allsavedir.joinpath(
                    f'PC{abwhichpcs[0]}-PC{abwhichpcs[1]}_transitions_separated.csv'), index_col=0)

                
                ############## BOOTSTRAP MANY TRAJECTORIES ##########
                bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                        rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ttot, #set the total bootstrap time
                        ntrans, #how many transitions to sample at each step
                        bsiter, #number of times to bootstrap
                        )


                ############# open average bootstrapped currents ###################
                bsfield_sep = DetailedBalance.get_avg_current_error(
                        bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ntrans, #how many transitions to sample at each step
                        )
    
                
                
                ############# measure aer and cycling frequencies ###########
                #add specific scaling
                xyscaling = [centers[f'PC{abwhichpcs[0]}'].diff().mean(),centers[f'PC{abwhichpcs[1]}'].diff().mean()]
                #set the origin to the actual center
                center = allorigins[int(a-1)][int(b-(2+a-1))]

                DetailedBalance.get_aer_cf(
                    bstrans,
                    nbins, #how many bins in the x and y cgps axes
                    xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                    center, #origin in [x bin,y bin]
                    allsavedir, #where to save calculated aers and cfs
                    abwhichpcs, #which two PCs to use in the cgps [x,y]
                    ntrans, #how many transitions to sample at each step
                    )
                
                

                ############### measure aer and cycling frequency for the raw transitions
                #get the area scaling in x and y based on the size of the bins in the cgps
                results = []
                for i, cells in rawtrans.groupby('CellID'):
                    #sort data and get continuous transitions in order
                    cell = cell.sort_values('real_time').reset_index(drop = True)
                    #resets in cumulative time represent a change between non-consecutive
                    #series of interpolated transitions
                    diff = cell.cumulative_time.diff()
                    difflist = [0]
                    difflist.extend(diff[diff<=0].index.to_list())
                    if difflist[-1] < len(cell):
                        difflist.append(len(cell))
                    #make a list of lists with the indices of consecutive time points
                    runs = [list(range(difflist[x], difflist[x+1])) for x in range(len(difflist)-1)]
                    for r in runs:
                        cell = cells.iloc[r].reset_index(drop=True)
                        results.append(DetailedBalance.get_area_enclosing_rate((
                            cell,
                            nbins,
                            xyscaling,
                            center,
                            )))

                #make a dataframe and save it
                allaers = pd.concat(results, ignore_index = True)
                justaers = allaers[['CellID','cell','Treatment','aer','angular_velocity']].copy()
                justaers.to_csv(allsavedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Aaron\\Combined_37C_Confocal_PCA_shape\\random\\allCGPS\\PC1-PC2_transitions_separated.csv'